<a href="https://colab.research.google.com/github/smerarawal/F1-Pit-Stop-Strategy-Agent/blob/main/RL_Pit_Stop_Strategy_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

df = pd.read_csv("train.csv", low_memory=False)

# Example (adjust column name if needed)
# df['position_change'] = df['final_position'] - df['grid_position']

print("Available columns:", df.columns.tolist())

Available columns: ['race_id', 'season', 'race_round', 'circuit_id', 'circuit_country', 'circuit_length_km', 'circuit_turns', 'driver_id', 'driver_first_name', 'driver_last_name', 'driver_nationality', 'driver_skill_rating', 'driver_aggression_rating', 'driver_consistency_rating', 'team_id', 'team_budget_tier', 'team_car_speed_rating', 'team_car_downforce_rating', 'grid_position', 'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'lap', 'position', 'lap_time_sec', 's1_time_sec', 's2_time_sec', 's3_time_sec', 'tire_compound', 'tire_age_laps', 'tire_wear_pct', 'fuel_load_kg', 'ers_deploy_pct', 'ers_harvest_pct', 'drs_activated', 'gap_to_leader_sec', 'gap_ahead_sec', 'gap_behind_sec', 'track_status', 'weather_current', 'race_weather_start', 'race_air_temp_c', 'race_track_temp_c', 'pit_stop_this_lap', 'pit_stop_duration_sec', 'pit_new_compound', 'finish_position', 'status', 'points', 'fastest_lap', 'race_race_name', 'race_race_date', 'race_race_start_time', 'race_weather_end', 'race_humidity_pc

In [4]:
!pip install gymnasium stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 39.4 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.3.0
    Uninstalling gymnasium-1.3.0:
      Successfully uninstalled gymnasium-1.3.0


In [21]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd

class F1Env(gym.Env):
    def __init__(self, df):
        super(F1Env, self).__init__()

        self.df = df.sort_values(['race_id','driver_id','lap'])
        self.races = self.df.groupby(['race_id','driver_id'])
        self.race_keys = list(self.races.groups.keys())

        # Action: 0 = stay out, 1 = pit
        self.action_space = spaces.Discrete(2)

        # State space (normalize approx ranges)
        self.observation_space = spaces.Box(
            low=0, high=1, shape=(8,), dtype=np.float32
        )

        self.current_race = None
        self.current_step = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.current_race = self.races.get_group(
            self.race_keys[np.random.randint(len(self.race_keys))]
        ).reset_index(drop=True)

        self.current_step = 0

        return self._get_state(), {}

    def _get_state(self):
        row = self.current_race.iloc[self.current_step]

        state = np.array([
            row['lap'] / 100,
            row['tire_wear_pct'] / 100,
            row['fuel_load_kg'] / 110,
            row['gap_to_leader_sec'] / 100,
            row['gap_ahead_sec'] / 10,
            row['gap_behind_sec'] / 10,
            row['race_track_temp_c'] / 60,
            row['ers_deploy_pct'] / 100
        ], dtype=np.float32)

        return state

    def step(self, action):
        # Get the current row for this step before advancing
        row_for_this_step = self.current_race.iloc[self.current_step]

        reward = 0
        reward -= row_for_this_step['lap_time_sec'] / 100

        # Handle PIT logic and its effect on *this step's reward*
        effective_tire_wear_for_reward = row_for_this_step['tire_wear_pct']
        if action == 1:
            reward -= 2  # pit penalty
            effective_tire_wear_for_reward = 10  # Reset for reward calculation in this step
            # Note: This does not persist the tire wear reset to the environment state.
            # If persistent change is desired, self.current_race would need to be modified.

        reward -= effective_tire_wear_for_reward / 50 # Changed from 100 to 50

        # Add reward for position change
        if self.current_step > 0:
            prev_pos = self.current_race.iloc[self.current_step - 1]['position']
            curr_pos = row_for_this_step['position']
            reward += (prev_pos - curr_pos) * 0.5

        # Advance the step counter
        self.current_step += 1

        # Check if the episode is now done
        # The episode is done if self.current_step is past the last valid index
        done = self.current_step >= len(self.current_race)

        # Determine the next observation
        next_obs = None
        if done:
            # If done, there is no valid next state to read from the DataFrame.
            # Return the observation corresponding to the last valid step.
            temp_current_step = self.current_step
            self.current_step -= 1  # Temporarily set to the last valid row index
            next_obs = self._get_state()
            self.current_step = temp_current_step  # Restore for next reset()
        else:
            # If not done, get the observation for the new current step
            next_obs = self._get_state()

        return next_obs, reward, done, False, {}

In [20]:
df = pd.read_csv("train.csv")

# Keep only relevant columns
df = df[[
    'race_id','driver_id','lap',
    'tire_wear_pct','fuel_load_kg',
    'gap_to_leader_sec','gap_ahead_sec','gap_behind_sec',
    'race_track_temp_c','ers_deploy_pct',
    'lap_time_sec', 'position' # Added 'position' column
]]

df = df.dropna()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
from stable_baselines3 import PPO

env = F1Env(df)

model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64
)

model.learn(total_timesteps=50000)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 59       |
|    ep_rew_mean     | -242     |
| time/              |          |
|    fps             | 629      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 59.3        |
|    ep_rew_mean          | -240        |
| time/                   |             |
|    fps                  | 487         |
|    iterations           | 2           |
|    time_elapsed         | 8           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.013374343 |
|    clip_fraction        | 0.0149      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.688      |
|    explained_variance   | -0.00637    |
|    learning_rate        | 0.

In [10]:
obs, _ = env.reset()

total_reward = 0

for _ in range(200):
    action, _ = model.predict(obs)
    obs, reward, done, _, _ = env.step(action)
    total_reward += reward

    if done:
        break

print("Total Reward:", total_reward)

Total Reward: -161.13494000000003


In [11]:
def run_policy(env, policy_fn):
    obs, _ = env.reset()
    total_reward = 0

    for _ in range(300):
        action = policy_fn(obs)
        obs, reward, done, _, _ = env.step(action)
        total_reward += reward
        if done:
            break

    return total_reward

never_pit = lambda obs: 0

print("Never Pit:", run_policy(env, never_pit))

Never Pit: -195.94361999999998


In [12]:
always_pit = lambda obs: 1

print("Always Pit:", run_policy(env, always_pit))

Always Pit: -296.7629099999999


In [13]:
def smart_policy(obs):
    tire_wear = obs[1] * 100

    if tire_wear > 60:
        return 1
    return 0

print("Rule-based:", run_policy(env, smart_policy))

Rule-based: -259.84335


In [14]:
def agent_policy(obs):
    action, _ = model.predict(obs)
    return action

print("RL Agent:", run_policy(env, agent_policy))

RL Agent: -173.85109000000003


In [15]:
if action == 1:
    reward -= 2
    self.current_race.loc[self.current_step:, 'tire_wear_pct'] *= 0.3

In [19]:
model.learn(total_timesteps=100000)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 58.4     |
|    ep_rew_mean     | -202     |
| time/              |          |
|    fps             | 604      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 59.7          |
|    ep_rew_mean          | -207          |
| time/                   |               |
|    fps                  | 496           |
|    iterations           | 2             |
|    time_elapsed         | 8             |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 0.00083460537 |
|    clip_fraction        | 0.0104        |
|    clip_range           | 0.2           |
|    entropy_loss         | -0.12         |
|    explained_variance   | 0.836         |
